# Лекција 08 - Образац мулти-агент дизајна


## Постављање


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Зашто вишеагентски системи?

Задаће из стварног света као што је планирање путовања укључују много различитих врста стручности — логистику, локално познавање, буџетирање и друго. Један агент који покушава да управља свиме брзо постаје непрактичан.

Вишеагентски системи ово решавају кроз **специјализацију**: сваки агент се фокусира на једно подручје стручности, дајући квалитетније резултате него генералиста. Такође побољшавају **скалирање** — можете додати нове агенте (нпр. специјалисту за летове, критичара ресторана) без преписивања постојећег тока рада. Агенти се саставно повезују кроз структурирани пипелин, преносећи контекст од једног ка другом.


## Креирање специјализованих агената


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Креирање секвенцијалног тока рада

`WorkflowBuilder` вам омогућава да повежете агенте у усмерени графикон. Овде правимо једноставан двостепени процес: **TravelPlanner** саставља план пута, а затим га **TravelConcierge** прегледа и унапређује.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Додавање више агената у радни ток

Једна од највећих предности мулти-агентског обрасца је колико је лако проширити га. Испод додајемо агента **BudgetReviewer** који проверава план у односу на буџет путника, означава ставке које би могле да премаше трошковни лимит и предлаже алтернативе за уштеду новца. Радни ток сада покреће три агента у низу:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Резиме

У овој лекцији сте научили како да:

1. **Креирате специјализоване агенте** — сваки са фокусираним задатком (планирање, консијерж, преглед буџета).
2. **Повежете агенте у секвенцијални ток рада** користећи `WorkflowBuilder` и `add_edge`.
3. **Стримујете излаз** из пипелајна са више агената, пратећи који агент говори.
4. **Проширите ток рада** додавањем нових агената у ланац без измене постојећих.

Дизајн образац са више агената држи сваког агента једноставним, а истовремено производи богатије, темељније прегледане резултате него што било који појединачни агент може сам постићи.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Напомена**:
Овај документ је преведен коришћењем AI сервиса за превођење [Co-op Translator](https://github.com/Azure/co-op-translator). Иако се трудимо да превод буде прецизан, молимо вас имајте у виду да аутоматизовани преводи могу садржати грешке или нетачности. Оригинални документ на изворном језику треба сматрати ауторитетним извором. За важне информације препоручује се професионални превод од стране људи. Нисмо одговорни за било каква неспоразума или погрешна тумачења настала коришћењем овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
